In [1]:
import pandas as pd
import torch
from IPython.display import display

from doc_verify.baselines import clip_zero_shot, tfidf_baseline
from doc_verify.config import MODELS_DIR
from doc_verify.data import KIND_CLASSES, load_pages, source_dir
from doc_verify.evaluate import head_metrics
from doc_verify.rule_checker import rule_checker_predictions

pages = load_pages(('train', 'val'))          # the test split stays locked
train, val = pages[pages.split == 'train'], pages[pages.split == 'val']
out = MODELS_DIR / 'baselines'
out.mkdir(parents=True, exist_ok=True)
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(val.groupby('source').size().to_dict())

{'sad_syn': 450, 'tobacco3482': 522}


In [2]:
tfidf = tfidf_baseline(train, val)
tfidf.to_csv(out / 'tfidf_val_predictions.csv', index=False)
pd.Series(head_metrics(tfidf), name='TF-IDF + LR').round(3)

type_accuracy    0.747
type_macro_f1    0.706
auth_auc         0.485
auth_accuracy    0.498
auth_macro_f1    0.497
kind_macro_f1    0.133
Name: TF-IDF + LR, dtype: float64

In [3]:
sad_val = val[val.source == 'sad_syn']
rules = rule_checker_predictions(sad_val, source_dir('sad_syn'))
rules['kind_pred'] = rules.kind_name.map({name: i for i, name in enumerate(KIND_CLASSES)})
rules['type_label'] = -1
rules.to_csv(out / 'rules_val_predictions.csv', index=False)

display(pd.Series(head_metrics(rules), name='rule checker').round(3))
print(f"pages with every rule checkable: {(rules.rules_checked == rules.rules_total).mean():.1%}")
rules.groupby('doc_type').agg(
    every_rule_checkable=('rules_checked', lambda c: (c == rules.loc[c.index, 'rules_total']).mean()),
    false_alarm_on_clean=('auth_pred', lambda p: p[rules.loc[p.index, 'auth_label'] == 0].mean()),
    caught_fakes=('auth_pred', lambda p: p[rules.loc[p.index, 'auth_label'] == 1].mean()),
).round(3)

auth_auc         0.758
auth_accuracy    0.753
auth_macro_f1    0.753
kind_macro_f1    0.662
Name: rule checker, dtype: float64

pages with every rule checkable: 33.8%


,every_rule_checkable,false_alarm_on_clean,caught_fakes
doc_type,,,
bank_statement,0.427,0.387,0.800
offer_letter,0.533,0.293,0.853
transcript,0.053,0.160,0.707


In [4]:
readable = sad_val.set_index('page_index')['altered_value_ocr_ok']
fakes = rules[rules.auth_label == 1].assign(evidence_readable=lambda f: f.page_index.map(readable).astype(bool))
fakes.groupby(['doc_type', 'evidence_readable']).auth_pred.agg(['size', 'mean']).rename(columns={'size': 'fakes', 'mean': 'caught'}).round(3)

fakes  caught
doc_type       evidence_readable               
bank_statement False                 12   0.333
               True                  63   0.889
offer_letter   False                  8   0.750
               True                  67   0.866
transcript     False                  4   0.000
               True                  71   0.746

In [5]:
clip = clip_zero_shot(val[val.source == 'tobacco3482'], device=device)
clip.to_csv(out / 'clip_zero_shot_val_predictions.csv', index=False)
pd.Series(head_metrics(clip), name='CLIP zero-shot').round(3)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

type_accuracy    0.418
type_macro_f1    0.433
Name: CLIP zero-shot, dtype: float64

In [6]:
summary = pd.DataFrame({
    'TF-IDF + LR': head_metrics(tfidf),
    'rule checker': head_metrics(rules),
    'CLIP zero-shot': head_metrics(clip),
}).T
summary.to_csv(out / 'val_summary.csv')
summary.round(3)

,type_accuracy,type_macro_f1,auth_auc,auth_accuracy,auth_macro_f1,kind_macro_f1
TF-IDF + LR,0.747,0.706,0.485,0.498,0.497,0.133
rule checker,NaN,NaN,0.758,0.753,0.753,0.662
CLIP zero-shot,0.418,0.433,NaN,NaN,NaN,NaN
